## 0. Setup shared by the three validation notebooks
Paths are relative to the repository root. The statistics come from `tools/validation_metrics.py`
(Ministry of Transport guideline, draft 6, September 2024); results go to `Output/validation_mot/`.

In [1]:
import os, sys, warnings
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'
sys.path.insert(0, 'tools')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import validation_metrics as vm
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

BLUE, ORANGE, AQUA, PURPLE = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6'
INK, INK2, MUTED, GRID, AXIS = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
OUT = 'Output/validation_mot'; FIG = 'Output/figures'
os.makedirs(OUT, exist_ok=True); os.makedirs(FIG, exist_ok=True)

def style_ax(ax, title=None, xlabel=None, ylabel=None):
    ax.grid(True, color=GRID, linewidth=0.6); ax.set_axisbelow(True)
    for s in ax.spines.values(): s.set_color(AXIS)
    ax.tick_params(colors=INK2, labelsize=9)
    if title: ax.set_title(title, color=INK, fontsize=11)
    if xlabel: ax.set_xlabel(xlabel, color=INK2, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=INK2, fontsize=10)

def is_pointer(p): return open(p, 'rb').read(40).startswith(b'version https://git-lfs')
def need(*paths):
    for p in paths: assert not is_pointer(p), f'{p} is a Git LFS pointer: python3 tools/lfs_pull.py {p}'

SUMMARY = []      # one row per guideline check; written to Output/validation_mot/summary_<stage>.csv
def add_summary(test, ref, item, level, criterion, result, status, explanation, data):
    SUMMARY.append({'test': test, 'guideline': ref, 'item': item, 'level': level, 'criterion': criterion,
                    'result': result, 'status': status, 'explanation': explanation, 'data (vintage)': data})

def scatter_origin(ax, obs, mod, title, xlabel, ylabel, logscale=False, color=BLUE):
    """Guideline format: observed on x, model on y, regression through the origin with equation and R2."""
    o = np.asarray(obs, float); m = np.asarray(mod, float); ok = np.isfinite(o) & np.isfinite(m)
    o, m = o[ok], m[ok]
    ax.scatter(o, m, s=10, color=color, alpha=0.55, linewidths=0)
    hi = max(o.max(), m.max()) * 1.05 if len(o) else 1
    ax.plot([0, hi], [0, hi], color=MUTED, lw=0.8, ls=':')
    b = vm.slope_origin(o, m); r2 = vm.r2_correl(o, m)
    ax.plot([0, hi], [0, b * hi], color=ORANGE, lw=1.5)
    ax.text(0.04, 0.95, f'y = {b:.2f}x   R² = {r2:.2f}   n = {len(o)}', transform=ax.transAxes, va='top', fontsize=9, color=INK)
    if logscale: ax.set_xscale('symlog', linthresh=10); ax.set_yscale('symlog', linthresh=10)
    style_ax(ax, title, xlabel, ylabel)

def save_fig(fig, name):
    fig.savefig(f'{FIG}/{name}', dpi=150, bbox_inches='tight', facecolor='white'); plt.close(fig)

# Bus destination pattern: RavKav's own alightings or the on-board survey's probabilities, with RavKav as production only?

**Question.** What if the bus matrix took only RavKav's *boardings* (the production of each origin zone) and spread them over destinations with the
on-board survey's probabilities, instead of RavKav's own inferred alighting zone? That is the step-9 product (`Output/bus/bus_od_taz_new.csv`), and,
as the destination prior of the survey calibration, the version the chain used until 23 September 2026
(`bus_calibrated_onboard_prior_*`, kept). The two patterns share the same production, so the comparison isolates the destination pattern.

**Candidates (778 x 778 TAZ, AM 06:00-09:00, bus):**
* **A** RavKav May 2022 journeys on RavKav's own inferred alightings (`bus_od_taz_avg.csv`)
* **B** RavKav production x on-board probabilities (`bus_od_taz_new.csv`): the proposal
* **E** a hybrid: A at super-zone level (which super-zone the journey goes to), B only to split the destinations *inside* the destination super-zone
* **C** the current calibrated layer (survey + RavKav, prior A, 2018); **D** the same calibration with prior B (the earlier version)

**Independent evidence.** The survey's bus trips (days 1 and 2, both ends in the study area, AM; 1,790 sampled trips) are used by neither A, B nor E,
so they can judge the pattern of A, B and E without circularity (C and D contain the survey, so they are shown for completeness, not judged on it).
Tests: (1) how well each pattern predicts where the survey's bus trips from each origin super-zone go (destination super-zone, destination 1250-zone, and the 1250-zone split inside the
destination super-zone); (2) trip length against the survey's reported distance (KS); (3) the local share of trips.

In [2]:
TRIPS = 'Input/THS_2017-2018/trips_ths_2017.xlsx'; KEYS = 'Input/Matrices/1270_02_09_2021_TAZ_North_keys.csv'
need(TRIPS, KEYS, 'Input/TAZ_2636_Keys.xlsx')
from scipy.spatial.distance import jensenshannon
import geopandas as gpd
zon = pd.read_csv('Input/Zonal_2020.csv', encoding='cp1255').set_index('TAZ_ID')
TAZ = [int(c) for c in pd.read_csv('Output/ths2017/two_mode/car_taz.csv', index_col=0).columns]
keys_raw = pd.read_csv(KEYS, encoding='windows-1255')
_k = keys_raw[['TAZ_NUMBER', 'SZ_NEW']].dropna().astype(int).drop_duplicates('TAZ_NUMBER').set_index('TAZ_NUMBER')['SZ_NEW']
taz_sz = _k.reindex(TAZ).astype(int)                                  # the keys table the chain uses (decision of 4 Oct 2026)
kd = keys_raw[['TAZ_1270', 'TAZ_NUMBER']].dropna(subset=['TAZ_NUMBER']).astype({'TAZ_NUMBER': int}).drop_duplicates('TAZ_NUMBER')
taz_1250 = kd.set_index('TAZ_NUMBER')['TAZ_1270'].reindex(TAZ); Z1250 = sorted(taz_1250.unique())
d_ = pd.DataFrame({'z': taz_1250.values, 'g': taz_sz.values, 'pop': zon.loc[TAZ, 'POPULATION'].values}, index=TAZ)
sz_of_1250 = pd.Series({z: int(g.groupby('g')['pop'].sum().idxmax()) for z, g in d_.groupby('z')})
SZL = sorted(taz_sz.unique())

df = pd.read_excel(TRIPS); d = df.sort_values(['PerID3', 'SurveyDay', 'placeno']).copy()
g = d.groupby(['PerID3', 'SurveyDay']); d['origin'] = g['actTaz'].shift(1); d['trip_dep_h'] = g['Dep_h'].shift(1)
trips = d.dropna(subset=['origin', 'actTaz', 'trip_dep_h']).copy()
k26 = pd.read_excel('Input/TAZ_2636_Keys.xlsx').drop_duplicates('TAZ_2636').set_index('TAZ_2636')['TAZ_1250']
trips['o1250'] = trips['origin'].map(k26); trips['d1250'] = trips['actTaz'].map(k26)
B_ = trips[trips['o1250'].isin(Z1250) & trips['d1250'].isin(Z1250) & trips['trip_dep_h'].isin([6, 7, 8]) & trips['mainmode'].isin([3, 5])].copy()
B_['sz_o'] = B_['o1250'].map(sz_of_1250); B_['w'] = B_['new_wf'] / 2
print(f'survey bus trips, AM, both ends in the study area: {len(B_):,} sampled, {B_.w.sum():,.0f} expanded per day')

def lm(path):
    m = pd.read_csv(path, index_col=0); m.index = m.index.astype(int); m.columns = m.columns.astype(int)
    return m.reindex(index=TAZ, columns=TAZ, fill_value=0).fillna(0)
A = lm('Output/bus/bus_od_taz_avg.csv'); Bm = lm('Output/bus/bus_od_taz_new.csv')
C = lm('Output/ths2017/two_mode/bus_calibrated_taz.csv'); D = lm('Output/ths2017/two_mode/bus_calibrated_onboard_prior_taz.csv')
print('production (row sums) of A and B identical:', bool(np.allclose(A.sum(axis=1), Bm.sum(axis=1), rtol=1e-3, atol=1e-3)),
      f'| totals A {A.values.sum():,.0f}  B {Bm.values.sum():,.0f}  C {C.values.sum():,.0f}  D {D.values.sum():,.0f}')

# hybrid E: A decides the destination super-zone, B splits inside it
dest_sz = taz_sz.values; E = np.zeros((len(TAZ), len(TAZ)))
Av, Bv = A.values, Bm.values
orig_sz = taz_sz.values
for S_ in SZL:
    cols = np.where(dest_sz == S_)[0]
    a_tot = Av[:, cols].sum(axis=1)                                              # RavKav-alighting volume o -> super-zone S
    b_blk = Bv[:, cols]; b_tot = b_blk.sum(axis=1)
    share = np.zeros_like(b_blk)
    ok = b_tot > 0; share[ok] = b_blk[ok] / b_tot[ok, None]
    for i in np.where(~ok & (a_tot > 0))[0]:                                      # fall back to B pooled over the origin super-zone, then to A's own shares
        rows = np.where(orig_sz == orig_sz[i])[0]; pool = Bv[np.ix_(rows, cols)].sum(axis=0)
        share[i] = pool / pool.sum() if pool.sum() > 0 else (Av[i, cols] / Av[i, cols].sum())
    E[:, cols] = a_tot[:, None] * share
E = pd.DataFrame(E, index=TAZ, columns=TAZ)
assert np.allclose(E.sum(axis=1), A.sum(axis=1), rtol=1e-6, atol=1e-6), 'E keeps the production'

survey bus trips, AM, both ends in the study area: 1,790 sampled, 125,439 expanded per day


production (row sums) of A and B identical: False | totals A 92,440  B 92,713  C 115,430  D 127,185


## 1. Where do the survey's bus trips go? Prediction of the destination pattern by origin super-zone

In [3]:
Os = pd.get_dummies(pd.Series(taz_sz.values, index=TAZ)).reindex(columns=SZL, fill_value=0).astype(float).values
Dz = pd.get_dummies(pd.Series(taz_1250.values, index=TAZ)).reindex(columns=Z1250, fill_value=0).astype(float).values
dsz_of_z = np.array([sz_of_1250[z] for z in Z1250])
Ds = np.stack([(dsz_of_z == s).astype(float) for s in SZL], axis=1)               # 1250 zone -> destination super-zone (dominant)
def osz_z(X): return Os.T @ X.values @ Dz                                         # origin SZ x destination 1250-zone
Y = B_.groupby(['sz_o', 'd1250'])['w'].sum().unstack().reindex(index=SZL, columns=Z1250, fill_value=0).fillna(0).values
n_s = B_.groupby('sz_o').size().reindex(SZL).fillna(0).values
MIN_N = 20
CAND = {'A  RavKav own alightings': A, 'B  RavKav production x on-board': Bm, 'E  A to super-zone, B inside it': E, 'C  current calibrated layer (survey incl.)': C, 'D  on-board-prior calibrated layer (survey incl.)': D}
def jsd(p, q): return float(jensenshannon(p, q, base=2) ** 2)
def row_metrics(y, x, level):
    if level == 'destination super-zone':
        y, x = y @ Ds, x @ Ds
    elif level == 'split inside destination super-zone':       # conditional on the destination super-zone: weights = survey trips of that super-zone
        res = []; ws = []
        for j in range(len(SZL)):
            m = dsz_of_z == SZL[j]; yy, xx = y[m], x[m]
            if yy.sum() <= 0 or xx.sum() <= 0 or m.sum() < 2: continue
            res.append((jsd(yy / yy.sum(), xx / xx.sum()), vm.coincidence_ratio(yy, xx))); ws.append(yy.sum())
        if not ws: return np.nan, np.nan
        r = np.array(res); w = np.array(ws); return float((r[:, 0] * w).sum() / w.sum()), float((r[:, 1] * w).sum() / w.sum())
    if y.sum() <= 0 or x.sum() <= 0: return np.nan, np.nan
    return jsd(y / y.sum(), x / x.sum()), vm.coincidence_ratio(y, x)

rows = []; per_origin = []
use = [i for i in range(len(SZL)) if n_s[i] >= MIN_N]
for name, X in CAND.items():
    F = osz_z(X)
    for level in ['destination super-zone', 'destination 1250-zone', 'split inside destination super-zone']:
        vals = np.array([row_metrics(Y[i], F[i], level) for i in use]); w = np.array([Y[i].sum() for i in use])
        rows.append({'pattern': name, 'level': level, 'origin super-zones used': len(use), 'sampled trips behind them': int(n_s[use].sum()),
                     'JSD (trip-weighted, lower is better)': float(np.nansum(vals[:, 0] * w) / w[~np.isnan(vals[:, 0])].sum()),
                     'CR (trip-weighted, higher is better)': float(np.nansum(vals[:, 1] * w) / w[~np.isnan(vals[:, 1])].sum())})
        if level == 'destination 1250-zone':
            for i, v in zip(use, vals): per_origin.append({'pattern': name, 'origin super-zone': SZL[i], 'sampled trips': int(n_s[i]), 'JSD': v[0], 'CR': v[1]})
t1 = pd.DataFrame(rows); t1.to_csv(f'{OUT}/BusPattern_survey_destination_prediction.csv', index=False, float_format='%.4f')
po = pd.DataFrame(per_origin); po.to_csv(f'{OUT}/BusPattern_by_origin_superzone.csv', index=False, float_format='%.4f')
pd.set_option('display.max_colwidth', 60); print(t1.round(3).to_string(index=False))

                                          pattern                               level  origin super-zones used  sampled trips behind them  JSD (trip-weighted, lower is better)  CR (trip-weighted, higher is better)
                         A  RavKav own alightings              destination super-zone                       26                       1695                                 0.130                                 0.609
                         A  RavKav own alightings               destination 1250-zone                       26                       1695                                 0.365                                 0.328
                         A  RavKav own alightings split inside destination super-zone                       26                       1695                                 0.288                                 0.420
                  B  RavKav production x on-board              destination super-zone                       26                       1695       

In [4]:
# how often is each pattern the better predictor, origin by origin (1250-zone level), and the trip-weighted bootstrap of A minus B (households resampled)
wins = po.pivot(index='origin super-zone', columns='pattern', values='JSD')
a_, b_, e_ = 'A  RavKav own alightings', 'B  RavKav production x on-board', 'E  A to super-zone, B inside it'
print(f'origins where A beats B on JSD: {(wins[a_] < wins[b_]).sum()} of {len(wins)}; E beats A: {(wins[e_] < wins[a_]).sum()}; E beats B: {(wins[e_] < wins[b_]).sum()}')
rng = np.random.default_rng(7); hh = B_['HHID3'].values; uh = np.unique(hh)
idx_by_h = {h: np.where(hh == h)[0] for h in uh}
FA, FB, FE = osz_z(A), osz_z(Bm), osz_z(E)
oi = B_['sz_o'].map({s: i for i, s in enumerate(SZL)}).values; zi = B_['d1250'].map({z: i for i, z in enumerate(Z1250)}).values; wv = B_['w'].values
def pooled(F, y):
    num = den = 0.0
    for i in use:
        if y[i].sum() > 0: num += y[i].sum() * jsd(y[i] / y[i].sum(), F[i] / F[i].sum()); den += y[i].sum()
    return num / den
boot = []
for _ in range(200):
    pick = np.concatenate([idx_by_h[h] for h in rng.choice(uh, len(uh))])
    y = np.zeros_like(Y); np.add.at(y, (oi[pick], zi[pick]), wv[pick])
    boot.append((pooled(FA, y), pooled(FB, y), pooled(FE, y)))
boot = np.array(boot)
bs = pd.DataFrame({'JSD A': boot[:, 0], 'JSD B': boot[:, 1], 'JSD E': boot[:, 2]}); bs['B - A'] = bs['JSD B'] - bs['JSD A']; bs['A - E'] = bs['JSD A'] - bs['JSD E']
summ = bs.describe(percentiles=[.05, .95]).loc[['mean', '5%', '95%']].T; summ.to_csv(f'{OUT}/BusPattern_bootstrap.csv', float_format='%.4f'); print(summ.round(4).to_string())

origins where A beats B on JSD: 17 of 26; E beats A: 10; E beats B: 17


         mean      5%     95%
JSD A  0.4326  0.4129  0.4533
JSD B  0.5945  0.5702  0.6165
JSD E  0.4940  0.4752  0.5139
B - A  0.1619  0.1391  0.1875
A - E -0.0614 -0.0778 -0.0471


## 2. Trip length and local share against the survey's reported distance

In [5]:
shp = gpd.read_file('Input/TAZ_North/TAZ_North.shp').drop_duplicates('TAZ_NUMBER').set_index('TAZ_NUMBER').loc[TAZ]
cxy = shp.geometry.centroid; popv = zon.loc[TAZ, 'POPULATION'].clip(lower=0).values + 1e-9
zdf = pd.DataFrame({'z': taz_1250.values, 'x': cxy.x.values * popv, 'y': cxy.y.values * popv, 'w': popv, 'area': shp.geometry.area.values})
zc = zdf.groupby('z').agg(x=('x', 'sum'), y=('y', 'sum'), w=('w', 'sum'), area=('area', 'sum')).loc[Z1250]; zc['x'] /= zc['w']; zc['y'] /= zc['w']
X_ = zc[['x', 'y']].values; D1250 = np.sqrt(((X_[:, None, :] - X_[None, :, :]) ** 2).sum(-1)) / 1000.0; np.fill_diagonal(D1250, 0.52 * np.sqrt(zc['area'].values) / 1000.0)
ix = pd.Series(taz_1250.values, index=TAZ)
def agg1250(X): return pd.DataFrame(X.values, index=TAZ, columns=TAZ).groupby(ix.values).sum().T.groupby(ix.values).sum().T.reindex(index=Z1250, columns=Z1250, fill_value=0).values
Bd = B_[B_.TrvlDist.notna()]; n_eff = vm.kish_n(Bd.w)
def wmed(x, w):
    o = np.argsort(x); return float(x[o][np.searchsorted(np.cumsum(w[o]), 0.5 * w.sum())])
zpos = {z: i for i, z in enumerate(Z1250)}
st = B_[B_.o1250 != B_.d1250]; straight = np.array([D1250[zpos[a], zpos[b]] for a, b in zip(st.o1250, st.d1250)])
detour = float((st.TrvlDist / straight).median())
rows = [{'pattern': 'survey (reported distance)', 'median km': wmed(Bd.TrvlDist.values, Bd.w.values), 'mean km': float(np.average(Bd.TrvlDist, weights=Bd.w)),
         'share inside one 1250-zone': float(Bd.w[Bd.o1250 == Bd.d1250].sum() / Bd.w.sum()), 'share inside one super-zone': float(Bd.w[Bd.sz_o == Bd.d1250.map(sz_of_1250)].sum() / Bd.w.sum()), 'KS D': np.nan, 'critical (5 %)': np.nan, 'pass': np.nan}]
for name, X in CAND.items():
    M = agg1250(X); sel = M.ravel() > 0; xs, ws = (D1250 * detour).ravel()[sel], M.ravel()[sel]
    r = vm.ks_weighted(Bd.TrvlDist.values, Bd.w.values, xs, ws, n1=n_eff, n2=n_eff)
    loc_z = float(np.trace(M) / M.sum()); Msz = Ds.T @ M @ Ds; loc_s = float(np.trace(Msz) / Msz.sum())
    rows.append({'pattern': name, 'median km': wmed(xs, ws), 'mean km': float(np.average(xs, weights=ws)), 'share inside one 1250-zone': loc_z, 'share inside one super-zone': loc_s,
                 'KS D': r['D'], 'critical (5 %)': r['critical'], 'pass': r['pass']})
t2 = pd.DataFrame(rows); t2.to_csv(f'{OUT}/BusPattern_trip_length_and_local_share.csv', index=False, float_format='%.4f'); print(f'detour factor used {detour:.2f}'); print(t2.round(3).to_string(index=False))

detour factor used 0.99
                                          pattern  median km  mean km  share inside one 1250-zone  share inside one super-zone  KS D  critical (5 %)  pass
                       survey (reported distance)      2.905    6.391                       0.133                        0.509   NaN             NaN   NaN
                         A  RavKav own alightings      3.512    6.991                       0.111                        0.439 0.078           0.058 False
                  B  RavKav production x on-board      6.325   13.635                       0.054                        0.249 0.240           0.058 False
                  E  A to super-zone, B inside it      4.104    7.480                       0.079                        0.439 0.138           0.058 False
       C  current calibrated layer (survey incl.)      4.596    8.107                       0.084                        0.462 0.179           0.058 False
D  on-board-prior calibrated layer (survey inc

In [6]:
j = t1.set_index(['pattern', 'level'])
def get(p, l, c): return float(j.loc[(p, l), c])
add_summary('T7b', '3.5', 'destination pattern: A (RavKav alightings) vs B (RavKav production x on-board), survey bus trips as evidence', 'origin super-zone, destination 1250-zone',
            'lower JSD / higher CR against the survey',
            f'JSD A {get(a_, "destination 1250-zone", "JSD (trip-weighted, lower is better)"):.3f} vs B {get(b_, "destination 1250-zone", "JSD (trip-weighted, lower is better)"):.3f}; CR A {get(a_, "destination 1250-zone", "CR (trip-weighted, higher is better)"):.2f} vs B {get(b_, "destination 1250-zone", "CR (trip-weighted, higher is better)"):.2f}',
            'finding', 'Evidence for the choice of the destination pattern; the guideline sets no criterion for it.', 'survey bus trips 2017/18 (independent of A, B, E); RavKav May 2022; on-board survey')
add_summary('T7b', '3.5', 'split inside the destination super-zone: A vs B vs E', 'destination 1250-zone given destination super-zone', 'lower JSD / higher CR',
            f'JSD A {get(a_, "split inside destination super-zone", "JSD (trip-weighted, lower is better)"):.3f}, B {get(b_, "split inside destination super-zone", "JSD (trip-weighted, lower is better)"):.3f}, E {get(e_, "split inside destination super-zone", "JSD (trip-weighted, lower is better)"):.3f}',
            'finding', 'Isolates the fine pattern: does the on-board survey place the trips better inside a super-zone than RavKav\'s inferred alighting zone?', 'as above')
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), facecolor='white')
lv = ['destination super-zone', 'destination 1250-zone', 'split inside destination super-zone']; cols = {'A  RavKav own alightings': BLUE, 'B  RavKav production x on-board': ORANGE, 'E  A to super-zone, B inside it': AQUA}
for ax, metric, ttl in zip(axes, ['JSD (trip-weighted, lower is better)', 'CR (trip-weighted, higher is better)'], ['Jensen-Shannon divergence (lower is better)', 'Coincidence ratio (higher is better)']):
    for k_, (nm, c) in enumerate(cols.items()):
        ax.bar(np.arange(3) + (k_ - 1) * 0.27, [get(nm, l, metric) for l in lv], 0.26, color=c, label=nm)
    ax.set_xticks(range(3)); ax.set_xticklabels(['destination\nsuper-zone', 'destination\n1250-zone', 'split inside\nsuper-zone'], fontsize=9); style_ax(ax, ttl); ax.legend(frameon=False, fontsize=7)
save_fig(fig, 'mot_BusPattern_prediction.png')

## Summary of this notebook against the guideline

In [7]:
SUM = pd.DataFrame(SUMMARY); SUM.to_csv(f'{OUT}/summary_busPattern.csv', index=False)
SUM[['test', 'guideline', 'item', 'level', 'criterion', 'result', 'status']]

,test,guideline,item,level,criterion,result,status
0,T7b,3.5,destination pattern: A (RavKav alightings) vs B (RavKav ...,"origin super-zone, destination 1250-zone",lower JSD / higher CR against the survey,JSD A 0.365 vs B 0.540; CR A 0.33 vs B 0.20,finding
1,T7b,3.5,split inside the destination super-zone: A vs B vs E,destination 1250-zone given destination super-zone,lower JSD / higher CR,"JSD A 0.288, B 0.356, E 0.358",finding
